## OPs & Params

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import torch
import torch.nn as nn
import pickle
from compression.utils import replace_linear_with_, dyna_set_sparse_budget
from compression import create_compressed_deit_small, SparseQuantLinear, SparseLinearFrozen, SparseLinear, QuantMatmul
from dse.model_analysis import analysis_sparse_model, DeiTConfig

SPARSE_BUDGET_PATH = '/home/chenzhiqiang/MaskLLM-4V/deit_small_layerwise_importance_20251223.pkl'
sb = pickle.load(open(SPARSE_BUDGET_PATH, 'rb'))

model = create_compressed_deit_small(pretrained=False)
cfg = DeiTConfig(hidden_size=384, num_heads=6, prune_tokens=[0,] * 3, num_layers=12, num_classes=1000, prune_layers=[3, 6, 9])
ay = analysis_sparse_model(model, cfg, sb)

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from timm.models import create_model
model = create_model('deit_small_patch16_224', pretrained=True)
cfg = DeiTConfig(hidden_size=384, num_heads=6, prune_tokens=[0,] * 3, num_layers=12, num_classes=1000, prune_layers=[3, 6, 9])
ay = analysis_sparse_model(model, cfg, None)

In [4]:
ay

{'total_model_params': 21955432,
 'total_non_zero_params': 21955432,
 'total_dense_flops': 8490683368,
 'total_sparse_flops': 8490683368,
 'layer_metrics': {'patch_embed.proj': {'name': 'patch_embed.proj',
   'params': 295296,
   'non_zero_params': 295296,
   'dense_flops': 115680768,
   'sparse_flops': 115680768,
   'seq_length': 0,
   'input_shape': (1, 3, 224, 224)},
  'blocks.0.attn.qkv': {'name': 'blocks.0.attn.qkv',
   'params': 443520,
   'non_zero_params': 443520,
   'dense_flops': 174519936,
   'sparse_flops': 174519936,
   'seq_length': 197,
   'input_shape': (1, 197, 384)},
  'blocks.0.attn.proj': {'name': 'blocks.0.attn.proj',
   'params': 147840,
   'non_zero_params': 147840,
   'dense_flops': 58173312,
   'sparse_flops': 58173312,
   'seq_length': 197,
   'input_shape': (1, 197, 384)},
  'blocks.0.mlp.fc1': {'name': 'blocks.0.mlp.fc1',
   'params': 591360,
   'non_zero_params': 591360,
   'dense_flops': 232693248,
   'sparse_flops': 232693248,
   'seq_length': 197,
   'in

In [25]:
# print(ay)
for k, v in ay['layer_metrics'].items():
    if v['dense_flops'] > v['sparse_flops']:
        print(k, v)
    

blocks.0.attn.qkv {'name': 'blocks.0.attn.qkv', 'params': 443520, 'non_zero_params': 148638, 'dense_flops': 174519936, 'sparse_flops': 58336428, 'seq_length': 197, 'input_shape': (1, 197, 384)}
blocks.0.attn.proj {'name': 'blocks.0.attn.proj', 'params': 147840, 'non_zero_params': 147830, 'dense_flops': 58173312, 'sparse_flops': 58169372, 'seq_length': 197, 'input_shape': (1, 197, 384)}
blocks.0.mlp.fc1 {'name': 'blocks.0.mlp.fc1', 'params': 591360, 'non_zero_params': 591312, 'dense_flops': 232693248, 'sparse_flops': 232674336, 'seq_length': 197, 'input_shape': (1, 197, 384)}
blocks.0.mlp.fc2 {'name': 'blocks.0.mlp.fc2', 'params': 590208, 'non_zero_params': 147842, 'dense_flops': 232466304, 'sparse_flops': 58174100, 'seq_length': 197, 'input_shape': (1, 197, 1536)}
blocks.1.attn.qkv {'name': 'blocks.1.attn.qkv', 'params': 443520, 'non_zero_params': 443513, 'dense_flops': 174519936, 'sparse_flops': 174517178, 'seq_length': 197, 'input_shape': (1, 197, 384)}
blocks.1.attn.proj {'name': 'b